In [ ]:
from pymilvus import MilvusClient

client = MilvusClient("milvus_demo.db")


In [ ]:
"""
需要安装额外的库来支持多种文件格式:
pip install pymupdf python-docx python-pptx openpyxl pandas
"""

import re
import os
from typing import List, Dict, Any, Optional, Callable
from pathlib import Path


class TextSplitter:
    """
    文本分割器类，用于将长文本分割成较小的块
    支持多种分割策略和文件格式，包括PDF、Word、Excel、PPT等
    """
    
    # 预定义的分割规则
    PREDEFINED_RULES = {
        "default": ["\n\n", "\n", ". ", "。", "! ", "? ", "！", "？", ";", "；", ",", "，"],
        "paragraph": ["\n\n"],
        "sentence": [".", "。", "!", "！", "?", "？", ";", "；"],
        "line": ["\n"],
        "markdown": ["\n\n", "\n# ", "\n## ", "\n### ", "\n#### ", "\n##### ", "\n###### "],
        "python": ["\n\n", "\nimport ", "\nclass ", "\ndef ", "\nif __name__"],
        "java": ["\n\n", "\nimport ", "\nclass ", "\npublic ", "\nprivate ", "\nprotected "],
        "javascript": ["\n\n", "\nimport ", "\nclass ", "\nfunction ", "\nconst ", "\nlet ", "\nvar "]
    }

    def __init__(self, 
                 chunk_size: int = 1000, 
                 chunk_overlap: int = 200,
                 separators: Optional[List[str]] = None,
                 predefined_rule: Optional[str] = None,
                 length_function: Callable[[str], int] = len,
                 keep_separator: bool = False):
        """
        初始化文本分割器
        
        Args:
            chunk_size: 每个文本块的最大长度
            chunk_overlap: 文本块之间的重叠长度
            separators: 分割符列表，按优先级排序
            predefined_rule: 预定义分割规则名称
            length_function: 长度计算函数
            keep_separator: 是否保留分割符
        """
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.length_function = length_function
        self.keep_separator = keep_separator
        
        # 处理分割符
        if separators is not None:
            self.separators = separators
        elif predefined_rule is not None:
            self.separators = self.PREDEFINED_RULES.get(predefined_rule, self.PREDEFINED_RULES["default"])
        else:
            self.separators = self.PREDEFINED_RULES["default"]

    def split_text(self, text: str) -> List[str]:
        """
        将文本分割成较小的块
        
        Args:
            text: 待分割的文本
            
        Returns:
            分割后的文本块列表
        """
        if self.length_function(text) <= self.chunk_size:
            return [text]
            
        chunks = []
        start = 0
        
        while start < len(text):
            # 确定当前块的结束位置
            end = min(start + self.chunk_size, len(text))
            
            # 如果不是最后一块，尝试在分隔符处分割
            if end < len(text):
                # 寻找合适的分割点
                split_pos = self._find_split_position(text, start, end)
                if split_pos != -1:
                    end = split_pos
            
            # 提取文本块
            chunk = text[start:end].strip()
            if chunk:
                chunks.append(chunk)
            
            # 移动起始位置（考虑重叠）
            if end >= len(text):  # 已经处理完所有文本
                break
            elif self.length_function(text[start:end]) <= self.chunk_size:
                # 正常情况下按重叠长度移动
                start = max(start + 1, end - self.chunk_overlap)
            else:
                # 特殊情况处理
                start = end - self.chunk_overlap
                if start <= 0:
                    start = end
                
        return chunks

    def _find_split_position(self, text: str, start: int, end: int) -> int:
        """
        在指定范围内寻找最佳分割位置
        
        Args:
            text: 原始文本
            start: 起始位置
            end: 结束位置
            
        Returns:
            最佳分割位置，如果未找到返回-1
        """
        # 从后向前查找分割符
        for separator in self.separators:
            # 在当前块中查找分割符
            split_pos = text.rfind(separator, start, end)
            if split_pos != -1:
                if self.keep_separator:
                    return split_pos + len(separator)
                else:
                    return split_pos  # 不保留分隔符
                
        return -1

    def split_document(self, 
                       content: str, 
                       metadata: Optional[Dict[str, Any]] = None) -> List[Dict[str, Any]]:
        """
        分割文档并保留元数据
        
        Args:
            content: 文档内容
            metadata: 文档元数据
            
        Returns:
            包含文本块和元数据的字典列表
        """
        if metadata is None:
            metadata = {}
            
        chunks = self.split_text(content)
        documents = []
        
        for i, chunk in enumerate(chunks):
            doc = {
                "content": chunk,
                "metadata": {
                    **metadata,
                    "chunk_index": i,
                    "total_chunks": len(chunks)
                }
            }
            documents.append(doc)
            
        return documents

    def split_documents(self, 
                        documents: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """
        批量分割文档
        
        Args:
            documents: 文档列表，每个文档包含content和metadata字段
            
        Returns:
            分割后的文档列表
        """
        split_docs = []
        for doc in documents:
            content = doc.get("content", "")
            metadata = doc.get("metadata", {})
            split_docs.extend(self.split_document(content, metadata))
            
        return split_docs

    def split_file(self, file_path: str, encoding: str = "utf-8") -> List[Dict[str, Any]]:
        """
        分割文件内容
        
        Args:
            file_path: 文件路径
            encoding: 文件编码
            
        Returns:
            分割后的文档列表
        """
        # 确保file_path是Path对象
        file_path = Path(file_path)
        
        # 检查文件是否存在
        if not file_path.exists():
            raise FileNotFoundError(f"文件 {file_path} 不存在")
            
        file_extension = file_path.suffix.lower()
        
        try:
            if file_extension == ".pdf":
                content = self._read_pdf_file(file_path)
            elif file_extension in [".docx", ".doc"]:
                content = self._read_docx_file(file_path)
            elif file_extension in [".pptx", ".ppt"]:
                content = self._read_pptx_file(file_path)
            elif file_extension in [".xlsx", ".xls"]:
                content = self._read_excel_file(file_path)
            elif file_extension == ".txt":
                content = self._read_text_file(file_path, encoding)
            elif file_extension in [".md", ".markdown"]:
                content = self._read_markdown_file(file_path, encoding)
            elif file_extension == ".json":
                content = self._read_json_file(file_path, encoding)
            else:
                # 默认按文本文件处理
                content = self._read_text_file(file_path, encoding)
                
            # 从文件路径提取元数据
            metadata = {
                "source": str(file_path),
                "file_type": file_extension[1:] if file_extension else "unknown"
            }
            
            return self.split_document(content, metadata)
        except Exception as e:
            raise Exception(f"读取文件 {file_path} 时出错: {str(e)}")

    def _read_text_file(self, file_path: Path, encoding: str) -> str:
        """读取文本文件"""
        with open(file_path, "r", encoding=encoding) as f:
            return f.read()

    def _read_pdf_file(self, file_path: Path) -> str:
        """读取PDF文件"""
        try:
            import fitz  # pymupdf
            doc = fitz.open(file_path)
            text_parts = []
            for page_num in range(len(doc)):
                page = doc.load_page(page_num)
                text_parts.append(page.get_text())
            doc.close()
            return "\n".join(text_parts)
        except ImportError:
            raise ImportError("需要安装pymupdf库来处理PDF文件: pip install pymupdf")

    def _read_docx_file(self, file_path: Path) -> str:
        """读取DOCX文件"""
        try:
            from docx import Document
            doc = Document(file_path)
            text = ""
            for paragraph in doc.paragraphs:
                text += paragraph.text + "\n"
            return text
        except ImportError:
            raise ImportError("需要安装python-docx库来处理DOCX文件: pip install python-docx")

    def _read_pptx_file(self, file_path: Path) -> str:
        """读取PPTX文件"""
        try:
            from pptx import Presentation
            prs = Presentation(file_path)
            text = ""
            for slide in prs.slides:
                for shape in slide.shapes:
                    if hasattr(shape, "text"):
                        text += shape.text + "\n"
            return text
        except ImportError:
            raise ImportError("需要安装python-pptx库来处理PPT文件: pip install python-pptx")

    def _read_excel_file(self, file_path: Path) -> str:
        """读取Excel文件"""
        try:
            import pandas as pd
            # 读取所有工作表
            excel_file = pd.ExcelFile(file_path)
            text = ""
            for sheet_name in excel_file.sheet_names:
                df = pd.read_excel(file_path, sheet_name=sheet_name)
                text += f"Sheet: {sheet_name}\n"
                text += df.to_string(index=False) + "\n\n"
            return text
        except ImportError:
            raise ImportError("需要安装openpyxl和pandas库来处理Excel文件: pip install openpyxl pandas")

    def _read_markdown_file(self, file_path: Path, encoding: str) -> str:
        """读取Markdown文件"""
        return self._read_text_file(file_path, encoding)

    def _read_json_file(self, file_path: Path, encoding: str) -> str:
        """读取JSON文件并转换为文本"""
        import json
        with open(file_path, "r", encoding=encoding) as f:
            data = json.load(f)
            return json.dumps(data, ensure_ascii=False, indent=2)

    def split_by_tokens(self, text: str, tokenizer: Any = None) -> List[str]:
        """
        基于token的分割方式（适用于大模型token限制）
        
        Args:
            text: 待分割文本
            tokenizer: 分词器，如果为None则使用简单分割
            
        Returns:
            分割后的文本块列表
        """
        if tokenizer is None:
            # 简单的基于空格和标点的分割
            tokens = re.findall(r'\S+|\s+', text)
        else:
            # 使用实际的分词器
            tokens = tokenizer.encode(text)
            
        chunks = []
        current_chunk = []
        current_length = 0
        
        for token in tokens:
            token_length = self.length_function(token)
            if current_length + token_length > self.chunk_size and current_chunk:
                # 创建当前块
                chunk_text = "".join(current_chunk)
                chunks.append(chunk_text.strip())
                
                # 根据重叠大小保留部分token
                if self.chunk_overlap > 0:
                    # 计算需要保留的token数量
                    overlap_tokens = []
                    overlap_length = 0
                    for t in reversed(current_chunk):
                        t_length = self.length_function(t)
                        if overlap_length + t_length <= self.chunk_overlap:
                            overlap_tokens.insert(0, t)
                            overlap_length += t_length
                        else:
                            break
                    current_chunk = overlap_tokens
                    current_length = overlap_length
                else:
                    current_chunk = []
                    current_length = 0
            
            current_chunk.append(token)
            current_length += token_length
            
        # 添加最后一个块
        if current_chunk:
            chunk_text = "".join(current_chunk)
            chunks.append(chunk_text.strip())
            
        return chunks

    def split_by_paragraphs(self, text: str) -> List[str]:
        """
        按段落分割文本
        
        Args:
            text: 待分割文本
            
        Returns:
            分割后的文本块列表
        """
        # 按双换行符分割段落
        paragraphs = re.split(r'\n\s*\n', text)
        chunks = []
        current_chunk = ""
        
        for paragraph in paragraphs:
            paragraph = paragraph.strip()
            if not paragraph:
                continue
                
            # 检查添加段落后是否会超出块大小
            test_chunk = (current_chunk + "\n\n" + paragraph) if current_chunk else paragraph
            if self.length_function(test_chunk) <= self.chunk_size:
                current_chunk = test_chunk
            else:
                # 如果当前块不为空，先保存
                if current_chunk:
                    chunks.append(current_chunk.strip())
                    
                # 如果段落本身就很长大于块大小，需要进一步分割
                if self.length_function(paragraph) > self.chunk_size:
                    # 使用常规分割方法处理大段落
                    sub_chunks = self.split_text(paragraph)
                    chunks.extend(sub_chunks)
                    current_chunk = ""
                else:
                    current_chunk = paragraph
                    
        # 添加最后一个块
        if current_chunk:
            chunks.append(current_chunk.strip())
            
        return chunks

    def split_by_semantic(self, text: str, sentence_transformer: Any = None) -> List[str]:
        """
        基于语义的分割方式（需要sentence transformers库）
        
        Args:
            text: 待分割文本
            sentence_transformer: 句子转换器模型
            
        Returns:
            分割后的文本块列表
        """
        try:
            import numpy as np
            from sklearn.metrics.pairwise import cosine_similarity
        except ImportError:
            raise ImportError("需要安装numpy和scikit-learn库: pip install numpy scikit-learn")
            
        if sentence_transformer is None:
            # 如果没有提供转换器，回退到按句子分割
            sentences = re.split(r'[.!?。！？]', text)
            sentences = [s.strip() for s in sentences if s.strip()]
        else:
            # 按句子分割
            sentences = re.split(r'[.!?。！？]', text)
            sentences = [s.strip() for s in sentences if s.strip()]
            
            # 获取句子嵌入
            embeddings = sentence_transformer.encode(sentences)
            
            # 计算相邻句子间的相似度
            similarities = []
            for i in range(len(embeddings) - 1):
                sim = cosine_similarity([embeddings[i]], [embeddings[i+1]])[0][0]
                similarities.append(sim)
            
            # 根据相似度决定分割点
            # 这里简化处理，实际可以更复杂
            
        # 根据句子长度分组
        chunks = []
        current_chunk = ""
        
        for sentence in sentences:
            test_chunk = (current_chunk + " " + sentence) if current_chunk else sentence
            if self.length_function(test_chunk) <= self.chunk_size:
                current_chunk = test_chunk
            else:
                if current_chunk:
                    chunks.append(current_chunk.strip())
                current_chunk = sentence
                
        if current_chunk:
            chunks.append(current_chunk.strip())
            
        return chunks

    def custom_split(self, text: str, split_function: Callable[[str], List[str]]) -> List[str]:
        """
        使用自定义分割函数
        
        Args:
            text: 待分割文本
            split_function: 自定义分割函数
            
        Returns:
            分割后的文本块列表
        """
        initial_splits = split_function(text)
        chunks = []
        
        for split in initial_splits:
            if self.length_function(split) <= self.chunk_size:
                chunks.append(split)
            else:
                # 如果分割后的部分仍然太大，继续使用基本分割方法
                sub_chunks = self.split_text(split)
                chunks.extend(sub_chunks)
                
        return chunks
        
    @classmethod
    def get_available_rules(cls) -> List[str]:
        """
        获取所有可用的预定义规则
        
        Returns:
            可用规则名称列表
        """
        return list(cls.PREDEFINED_RULES.keys())

In [44]:
"""
tina的Chromadb客户端支持
使用前需要安装chromadb，sentence-transformers
pip install chromadb sentence-transformers

"""
import chromadb,os
# from ..text_splitter import TextSplitter
from sentence_transformers import SentenceTransformer
import uuid
from typing import List, Dict, Any, Union


class ChromadbClient():
    def __init__(self, path: str, model_name: str = "all-MiniLM-L6-v2", name="demo_chromadb", 
                 text_splitter: TextSplitter = None):
        """
        初始化Chromadb客户端
        
        Args:
            path: 数据库存储路径
            model_name: 嵌入模型名称
            name: 集合名称
            text_splitter: 文本分割器实例
        """
        self.model = SentenceTransformer(model_name)
        self.client = chromadb.PersistentClient(path=path)
        self.collection = self.client.get_or_create_collection(
            name=name,
            metadata={"hnsw:space": "cosine"},
        )
        if text_splitter:
            self.text_splitter = text_splitter
        else:
            self.text_splitter = TextSplitter()
            
        # 设置最大批次大小
        self.max_batch_size = 5000
            
    def set_model(self, model: Any):
        """
        设置自定义模型
        
        Args:
            model: 自定义嵌入模型，需要有encode方法
        """
        self.model = model
        
    def add_documents(self, documents: List[Union[str, Dict]], metadatas: List[Dict] = None, 
                      ids: List[str] = None) -> int:
        """
        添加文档
        
        Args:
            documents: 文档列表，可以是字符串列表或包含content和metadata的字典列表
            metadatas: 文档元数据列表（可选）
            ids: 文档ID列表（可选）
            
        Returns:
            集合中的文档总数
        """
        # 处理不同格式的文档输入
        if isinstance(documents[0], dict):
            # 如果是字典格式，提取content
            contents = [doc.get("content", "") for doc in documents]
            if not metadatas:
                metadatas = [doc.get("metadata", {}) for doc in documents]
        else:
            # 如果是字符串格式
            contents = documents
            
        # 生成ID（如果没有提供）
        if not ids:
            ids = [str(uuid.uuid4()) for _ in range(len(documents))]
            
        # 分批处理以避免超过最大批次大小限制
        total_added = 0
        for i in range(0, len(contents), self.max_batch_size):
            batch_end = min(i + self.max_batch_size, len(contents))
            batch_contents = contents[i:batch_end]
            batch_metadatas = metadatas[i:batch_end] if metadatas else None
            batch_ids = ids[i:batch_end]
            
            # 生成嵌入
            embeddings = self.model.encode(batch_contents).tolist()
            
            # 添加到集合
            self.collection.add(
                documents=batch_contents,
                embeddings=embeddings,
                metadatas=batch_metadatas,
                ids=batch_ids
            )
            total_added += len(batch_contents)
            print(f"已添加 {total_added}/{len(contents)} 个文档")
            
        return self.collection.count()
        
    def add_file(self, file_path: str, file_metadata: Dict = None) -> int:
        """
        添加文件
        
        Args:
            file_path: 文件路径
            file_metadata: 文件元数据（可选）
            
        Returns:
            集合中的文档总数
        """
        # 使用文本分割器分割文件
        documents = self.text_splitter.split_file(file_path)
        
        # 添加文件元数据
        if file_metadata is None:
            file_metadata = {}
            
        for doc in documents:
            doc["metadata"].update(file_metadata)
            
        # 提取内容和元数据
        contents = [doc["content"] for doc in documents]
        metadatas = [doc["metadata"] for doc in documents]
        
        return self.add_documents(contents, metadatas)
        
    def query(self, query: str, n_results: int = 5, where: Dict = None, 
              where_document: Dict = None, prev_segments: int = 0, next_segments: int = 0) -> List[Dict]:
        """
        查询
        
        Args:
            query: 查询内容
            n_results: 返回结果数量
            where: 元数据过滤条件
            where_document: 文档内容过滤条件
            prev_segments: 返回前几个相邻片段
            next_segments: 返回后几个相邻片段
            
        Returns:
            查询结果列表，每个元素包含ids、content、confidence、source、chunk_index等信息
        """
        # 生成查询嵌入
        query_embedding = self.model.encode([query]).tolist()
        
        # 执行查询
        results = self.collection.query(
            query_embeddings=query_embedding,
            n_results=n_results,
            where=where,
            where_document=where_document,
            include=['documents', 'distances', 'metadatas']
        )
        
        # 构建返回结果
        formatted_results = []
        
        if results['ids'] and results['ids'][0]:
            for i in range(len(results['ids'][0])):
                # 计算置信度 (1 - distance)
                confidence = max(0, min(1, 1 - results['distances'][0][i]))
                
                # 基本信息
                result_item = {
                    "id": results['ids'][0][i],
                    "content": results['documents'][0][i],
                    "confidence": confidence,
                    "source": results['metadatas'][0][i].get('source', ''),
                    "chunk_index": results['metadatas'][0][i].get('chunk_index', 0),
                    "total_chunks": results['metadatas'][0][i].get('total_chunks', 0)
                }
                
                # 如果需要前后文段
                if (prev_segments > 0 or next_segments > 0) and results['metadatas'][0][i].get('source'):
                    # 获取相邻片段
                    adjacent_segments = self._get_adjacent_segments(
                        results['metadatas'][0][i], 
                        prev_segments, 
                        next_segments
                    )
                    result_item.update(adjacent_segments)
                
                formatted_results.append(result_item)
                
        return formatted_results
        
    def _get_adjacent_segments(self, metadata: Dict, prev_count: int, next_count: int) -> Dict:
        """
        获取相邻的文段片段
        
        Args:
            metadata: 当前文段的元数据
            prev_count: 前几个片段
            next_count: 后几个片段
            
        Returns:
            包含前后文段的字典
        """
        current_index = metadata.get('chunk_index', 0)
        source = metadata.get('source', '')
        total_chunks = metadata.get('total_chunks', 0)
        
        adjacent_segments = {}
        
        # 构造查询条件以获取同一文档的所有片段
        where_clause = {"source": source}
        
        try:
            # 获取同一文档的所有片段
            all_segments = self.collection.get(
                where=where_clause,
                include=['documents', 'metadatas']
            )
            
            # 按chunk_index排序
            if all_segments and all_segments.get('metadatas'):
                # 创建索引映射
                indexed_segments = {}
                for i in range(len(all_segments['ids'])):
                    chunk_idx = all_segments['metadatas'][i].get('chunk_index')
                    if chunk_idx is not None:
                        indexed_segments[chunk_idx] = {
                            'id': all_segments['ids'][i],
                            'content': all_segments['documents'][i],
                            'metadata': all_segments['metadatas'][i]
                        }
                
                # 获取前几个片段
                if prev_count > 0:
                    prev_segments_list = []
                    for i in range(max(0, current_index - prev_count), current_index):
                        if i in indexed_segments:
                            prev_segments_list.append(indexed_segments[i]['content'])
                    adjacent_segments['prev_segments'] = prev_segments_list
                
                # 获取后几个片段
                if next_count > 0:
                    next_segments_list = []
                    for i in range(current_index + 1, min(total_chunks, current_index + 1 + next_count)):
                        if i in indexed_segments:
                            next_segments_list.append(indexed_segments[i]['content'])
                    adjacent_segments['next_segments'] = next_segments_list
                    
        except Exception as e:
            print(f"获取相邻文段时出错: {e}")
            adjacent_segments['prev_segments'] = []
            adjacent_segments['next_segments'] = []
            
        return adjacent_segments
        
    def get_documents_by_ids(self, ids: List[str]) -> List[Dict]:
        """
        根据文档ID获取对应的文段片段
        
        Args:
            ids: 文档ID列表
            
        Returns:
            文档信息列表
        """
        try:
            # 分批处理以避免超过最大批次大小限制
            all_documents = []
            for i in range(0, len(ids), self.max_batch_size):
                batch_ids = ids[i:i+self.max_batch_size]
                results = self.collection.get(
                    ids=batch_ids,
                    include=['documents', 'metadatas']
                )
                
                if results and results['ids']:
                    for j in range(len(results['ids'])):
                        doc = {
                            "id": results['ids'][j],
                            "content": results['documents'][j],
                            "source": results['metadatas'][j].get('source', ''),
                            "chunk_index": results['metadatas'][j].get('chunk_index', 0),
                            "total_chunks": results['metadatas'][j].get('total_chunks', 0)
                        }
                        all_documents.append(doc)
                        
            return all_documents
        except Exception as e:
            print(f"根据ID获取文档时出错: {e}")
            return []
        
    def delete(self, ids: List[str] = None, where: Dict = None, 
               where_document: Dict = None) -> bool:
        """
        删除文档
        
        Args:
            ids: 要删除的文档ID列表
            where: 元数据过滤条件
            where_document: 文档内容过滤条件
            
        Returns:
            删除是否成功
        """
        try:
            # 分批删除以避免超过最大批次大小限制
            if ids:
                for i in range(0, len(ids), self.max_batch_size):
                    batch_ids = ids[i:i+self.max_batch_size]
                    self.collection.delete(ids=batch_ids)
            else:
                self.collection.delete(
                    where=where,
                    where_document=where_document
                )
            return True
        except Exception as e:
            print(f"删除文档时出错: {e}")
            return False
            
    def get_document_count(self) -> int:
        """
        获取文档总数
        
        Returns:
            集合中的文档总数
        """
        return self.collection.count()
        
    def get_collection_info(self) -> Dict:
        """
        获取集合信息
        
        Returns:
            集合信息
        """
        return {
            "name": self.collection.name,
            "count": self.collection.count(),
            "metadata": self.collection.metadata
        }
        
    def reset_collection(self) -> bool:
        """
        重置集合（删除所有文档）
        
        Returns:
            重置是否成功
        """
        try:
            # 分批删除所有文档以避免超过最大批次大小限制
            while True:
                # 获取一批文档ID
                batch = self.collection.get(include=['documents'], limit=self.max_batch_size)
                if not batch or not batch.get('ids'):
                    break
                    
                # 删除这一批文档
                self.collection.delete(ids=batch['ids'])
                
                # 如果返回的文档数少于批次大小，说明已经删除完所有文档
                if len(batch['ids']) < self.max_batch_size:
                    break
                    
            return True
        except Exception as e:
            print(f"重置集合时出错: {e}")
            return False

In [ ]:
cdclient = ChromadbClient(
    path="test",
    model_name="BAAI/bge-small-zh"
)

In [ ]:
from modelscope import snapshot_download
from sentence_transformers import SentenceTransformer

model_dir = snapshot_download("damo/nlp_corom_sentence-embedding_chinese-base")
model = SentenceTransformer(model_dir)

In [45]:
client = ChromadbClient(
    path="test",
    model_name=model_dir
)
client.reset_collection()

No sentence-transformers model found with name C:\Users\qiqi\.cache\modelscope\hub\models\damo\nlp_corom_sentence-embedding_chinese-base. Creating a new one with mean pooling.
Some weights of BertModel were not initialized from the model checkpoint at C:\Users\qiqi\.cache\modelscope\hub\models\damo\nlp_corom_sentence-embedding_chinese-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


True

In [46]:
client.add_file(file_path="2504.19467v2.pdf")

已添加 5000/21711 个文档
已添加 10000/21711 个文档
已添加 15000/21711 个文档
已添加 20000/21711 个文档
已添加 21711/21711 个文档


21711

In [49]:
result = client.query(query="脑梗")

In [50]:
print(result)

[{'id': '1e53b67d-8314-43a0-b7c4-81c2ecd722ce', 'content': 'ications.', 'confidence': 0.5466251373291016, 'source': '2504.19467v2.pdf', 'chunk_index': 3337, 'total_chunks': 21711}, {'id': '035acd76-eeac-4e16-82d3-af16f7eac713', 'content': '7499 (2024).', 'confidence': 0.5266726613044739, 'source': '2504.19467v2.pdf', 'chunk_index': 6153, 'total_chunks': 21711}, {'id': '3a40d770-a44b-442c-8bb3-8f55e104546d', 'content': '00', 'confidence': 0.5217784643173218, 'source': '2504.19467v2.pdf', 'chunk_index': 2742, 'total_chunks': 21711}, {'id': 'f9829f7d-7047-4ad7-ada5-4342b58b335a', 'content': '99 (2024).', 'confidence': 0.5209243297576904, 'source': '2504.19467v2.pdf', 'chunk_index': 6155, 'total_chunks': 21711}, {'id': '4663ff19-6591-420a-adb1-effff21cda8c', 'content': 'ic', 'confidence': 0.5148007869720459, 'source': '2504.19467v2.pdf', 'chunk_index': 2945, 'total_chunks': 21711}]


In [ ]:
tx = TextSplitter()

In [ ]:
tx.split_file("README.md")

In [ ]:
tx.split_file("2504.19467v2.pdf")

In [3]:
import asyncio

async def main():
    print("Start")
    await asyncio.sleep(1)
    print("End")

await main()

Start
End
